# 🛡️ INTERCEPT CDSS — Motor de Decisão Clínica v2
> **Clinical Decision Support System** | Prevenção de Interações Medicamentosas Adversas
>
> Baseado em: SBC 2023 · SBD 2024 · SBPT · ANVISA · AHA · FDA

In [1]:
# ── CÉLULA 1: Instalação das dependências ──────────────────────────────────
# Execute apenas uma vez por sessão
!pip install -q transformers torch

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\Ricardo\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\torch\\include\\ATen\\native\\transformers\\cuda\\mem_eff_attention\\iterators\\predicated_tile_access_iterator_residual_last.h'


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Ricardo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# ── CÉLULA 2: Carregamento do motor INTERCEPT ──────────────────────────────
import sys
import os

# Detecta se está no Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # No Colab: faz clone do repositório e adiciona ao path
    if not os.path.exists('/content/Nesis.AI'):
        !git clone https://github.com/gabrielbringel/Nesis.AI.git /content/Nesis.AI
    sys.path.insert(0, '/content/Nesis.AI')
else:
    # Local: adiciona o diretório do notebook ao path
    sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from intercept_engine import InterceptCDSS, display_resultado

print('✅ Motor INTERCEPT CDSS carregado com sucesso!')
print('   • intercept_rules.json → 30 regras clínicas')
print('   • intercept_drugs.json → 100+ fármacos mapeados')

In [ ]:
# ── CÉLULA 3: Carregamento do BioBERTpt (NER) ──────────────────────────────
# O NER extrai entidades do texto livre. O INTERCEPT usa fallback
# por dicionário+fuzzy se o NER retornar vazio, garantindo 100% de cobertura.

from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print(f'🖥️ Dispositivo: {"GPU" if device == 0 else "CPU"}')
print('⏳ Carregando BioBERTpt (pucpr/clinicalnerpt-medical)...')

ner_pipeline = pipeline(
    'ner',
    model='pucpr/clinicalnerpt-medical',
    aggregation_strategy='simple',
    device=device
)

# Inicializa o INTERCEPT com o NER
cdss = InterceptCDSS(ner_pipeline=ner_pipeline)

print('\n✅ BioBERTpt carregado! INTERCEPT pronto para análise.')

---
## 🧪 Caso Clínico 1 — Hipercalemia Grave
**Cenário:** Paciente hipertenso em uso de IECA. Médico tenta prescrever Espironolactona (diurético poupador de K+).  
**Risco:** Hipercalemia com potencial arritmia fatal.

In [ ]:
# ── CASO 1: Hipercalemia ───────────────────────────────────────────────────
prontuario_1 = """
S: Paciente masculino, 65 anos, comparece a UBS com queixa de tosse e dores no corpo.
Relata HAS de longa data e DM2. Nega alergias conhecidas.
Em uso contínuo de enalapriu 20mg 1x/dia, aas infantil 100mg e metformina 850mg.
O: PA 150x90 mmHg. FC 82 bpm. Saturação 97%. Corado, hidratado.
A: Mialgia. PA descompensada. Tosse possivelmente relacionada ao IECA.
P: Considerar ajuste anti-hipertensivo. Paciente solicita algo para as dores.
   Médico considera prescrever Espironolactona 25mg para controle da PA
   e Ibuprofeno 400mg para as dores musculares.
"""

resultado_1 = cdss.analisar_prontuario(prontuario_1)
display_resultado(resultado_1)

---
## 🧪 Caso Clínico 2 — Asma + Betabloqueador Não-Seletivo
**Cenário:** Paciente asmático com HAS e taquicardia. Médico tenta prescrever Propranolol.  
**Risco:** Broncoespasmo severo por bloqueio β2 pulmonar.

In [ ]:
# ── CASO 2: Asma + Betabloqueador Não-Seletivo ────────────────────────────
prontuario_2 = """
S: Paciente feminina, 42 anos. Histórico de asma brônquica moderada em uso de Salbutamol
spray e Beclometasona inalatória. Procura UPA com queixa de palpitações e PA elevada.
Nega uso de outros medicamentos. Nega tabagismo.
O: PA 165x100 mmHg. FC 112 bpm. Ausculta pulmonar com leve sibilância.
A: Taquicardia sinusal + HAS estágio 2 em paciente asmática.
P: Médico considera iniciar Propranolol 40mg 2x/dia para controle de FC e PA.
"""

resultado_2 = cdss.analisar_prontuario(prontuario_2)
display_resultado(resultado_2)

---
## 🧪 Caso Clínico 3 — DM2 + Doença Renal + Anticoagulação
**Cenário:** Paciente diabético com DRC e em uso de Varfarina. Médico tenta prescrever Metformina e Ibuprofeno.  
**Riscos múltiplos:** Acidose lática (Metformina+DRC) + Hemorragia grave (Ibuprofeno+Varfarina).

In [ ]:
# ── CASO 3: Múltiplos alertas ─────────────────────────────────────────────
prontuario_3 = """
S: Paciente masculino, 72 anos. Diabético tipo 2 há 15 anos. Portador de insuficiência
renal crônica (creatinina 2,8 mg/dL, TFG estimada 28 mL/min). Em uso de varfarina 5mg
para FA e atorvastatina 40mg. Queixa de dor articular intensa nos joelhos (gota?).
O: PA 138x88. FC 74 bpm. INR = 2,8. Ureia 98. Creatinina 2,8.
A: Artralgia possivelmente gotosa. DRC G4. Anticoagulação em faixa terapêutica.
P: Médico considera prescrever metformina 850mg para controle glicêmico e
   ibuprofeno 600mg para dor articular.
"""

resultado_3 = cdss.analisar_prontuario(prontuario_3)
display_resultado(resultado_3)

---
## 🔬 Análise Livre — Digite seu próprio prontuário

In [ ]:
# ── ANÁLISE LIVRE ─────────────────────────────────────────────────────────
# Substitua o texto abaixo pelo prontuário que deseja analisar

meu_prontuario = """
Escreva o prontuário aqui...
Inclua: medicamentos em uso, diagnósticos/condições clínicas,
e os medicamentos que você quer prescrever.
"""

resultado = cdss.analisar_prontuario(meu_prontuario)
display_resultado(resultado)